# ComfyUI en Google Colab + Google Drive

Este notebook instala ComfyUI en el almacenamiento temporal de Colab y conserva modelos, LoRAs y resultados en Google Drive. Así se evita instalar todo el repositorio directamente en Drive, que puede ser lento.

**Uso:** ejecuta las celdas en orden. Para usar una GPU, abre **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. Las sesiones de Colab pueden desconectarse y tienen límites de uso.

**Seguridad:** nunca escribas tokens en una celda ni los compartas. Configúralos en **Colab → icono de llave (Secrets)** con los nombres `CIVITAI_TOKEN` y/o `HF_TOKEN` si los necesitas.


In [ ]:
# 1. Comprobar GPU y montar Google Drive
import os
import torch
from google.colab import drive

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: no hay GPU activa. En Colab, selecciona Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU.")

drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/ComfyUI"
DRIVE_MODELS = os.path.join(DRIVE_BASE, "models")
DRIVE_OUTPUT = os.path.join(DRIVE_BASE, "output")
os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print("Carpeta persistente:", DRIVE_BASE)


In [ ]:
# 2. Instalar ComfyUI en el disco temporal de Colab
# El código de ComfyUI va en /content; modelos y resultados se conservan en Drive.
import os
import subprocess
import shutil

repo = "/content/ComfyUI"
if not os.path.isdir(os.path.join(repo, ".git")):
    subprocess.run(
        ["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", repo],
        check=True
    )
else:
    subprocess.run(["git", "-C", repo, "pull"], check=True)

subprocess.run(
    ["pip", "install", "-q", "-r", os.path.join(repo, "requirements.txt")],
    check=True
)

# Crear carpetas persistentes en Drive y enlazarlas con ComfyUI.
model_types = ["checkpoints", "loras", "vae", "controlnet", "unet", "clip", "embeddings", "diffusion_models"]
for kind in model_types:
    drive_dir = os.path.join(DRIVE_MODELS, kind)
    local_dir = os.path.join(repo, "models", kind)
    os.makedirs(drive_dir, exist_ok=True)
    if os.path.islink(local_dir):
        os.unlink(local_dir)
    elif os.path.isdir(local_dir):
        for item in os.listdir(local_dir):
            src, dst = os.path.join(local_dir, item), os.path.join(drive_dir, item)
            if not os.path.exists(dst):
                try:
                    shutil.move(src, dst)
                except OSError:
                    pass
        shutil.rmtree(local_dir, ignore_errors=True)
    elif os.path.exists(local_dir):
        os.remove(local_dir)
    os.symlink(drive_dir, local_dir)

# Guardar las imágenes generadas en Drive.
local_output = os.path.join(repo, "output")
if os.path.islink(local_output):
    os.unlink(local_output)
elif os.path.isdir(local_output):
    for item in os.listdir(local_output):
        src, dst = os.path.join(local_output, item), os.path.join(DRIVE_OUTPUT, item)
        if not os.path.exists(dst):
            try:
                shutil.move(src, dst)
            except OSError:
                pass
    shutil.rmtree(local_output, ignore_errors=True)
os.symlink(DRIVE_OUTPUT, local_output)

print("Instalación y carpetas persistentes preparadas.")


## 3. Descargar modelos y LoRAs

Pega una URL directa al archivo de descarga del recurso. En Civitai suele ser una URL de descarga de modelo; en Hugging Face, usa el enlace directo al archivo (`resolve/main/...`) o una URL de descarga autorizada.

Para recursos privados, crea un secreto en Colab desde el icono de llave:
- `CIVITAI_TOKEN` para Civitai
- `HF_TOKEN` para Hugging Face

No todos los enlaces necesitan token. Respeta las licencias y condiciones del proveedor.


In [ ]:
# 3. Formulario de descarga de modelos y LoRAs
import os
import re
import subprocess
import ipywidgets as widgets
from IPython.display import display
from google.colab import userdata

provider = widgets.Dropdown(
    options=["URL directa", "Civitai", "Hugging Face"],
    value="URL directa",
    description="Proveedor:"
)
resource_type = widgets.Dropdown(
    options=[
        ("Modelo base / checkpoint", "checkpoints"),
        ("LoRA", "loras"),
        ("VAE", "vae"),
        ("ControlNet", "controlnet"),
        ("UNet / diffusion model", "unet"),
        ("CLIP", "clip"),
        ("Embeddings", "embeddings")
    ],
    value="checkpoints",
    description="Tipo:"
)
url_field = widgets.Text(
    placeholder="Pega aquí la URL directa de descarga",
    description="URL:",
    layout=widgets.Layout(width="100%")
)
filename_field = widgets.Text(
    placeholder="Opcional; ej. modelo.safetensors",
    description="Archivo:",
    layout=widgets.Layout(width="100%")
)
download_btn = widgets.Button(description="Descargar recurso", button_style="primary", icon="download")
download_out = widgets.Output()

def safe_filename(name):
    name = os.path.basename(name.strip())
    if not name or name in (".", "..") or not re.fullmatch(r"[\w.,()+ -]+", name):
        raise ValueError("Nombre de archivo no válido. Usa letras, números, punto, guion o guion bajo.")
    return name

def get_secret(secret_name):
    try:
        return userdata.get(secret_name)
    except Exception:
        return None

def on_download_clicked(_):
    with download_out:
        download_out.clear_output()
        url = url_field.value.strip()
        if not url.startswith(("https://", "http://")):
            print("Error: introduce una URL http:// o https:// válida.")
            return
        kind = resource_type.value
        dest_dir = os.path.join(DRIVE_MODELS, kind)
        os.makedirs(dest_dir, exist_ok=True)
        suggested = filename_field.value.strip()
        if suggested:
            try:
                filename = safe_filename(suggested)
            except ValueError as e:
                print(e)
                return
        else:
            # Es mejor especificar el nombre cuando el proveedor usa URLs sin nombre de archivo.
            from urllib.parse import urlparse, unquote
            filename = os.path.basename(unquote(urlparse(url).path)) or ""
            if not filename or filename in ("download", "api"):
                print("No se pudo deducir el nombre del archivo. Escríbelo en el campo Archivo.")
                return
            try:
                filename = safe_filename(filename)
            except ValueError as e:
                print(e)
                return

        dest = os.path.join(dest_dir, filename)
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            print("Ya existe un archivo con ese nombre:", dest)
            print("No se sobrescribió. Cambia el nombre o elimina el archivo existente manualmente si quieres reemplazarlo.")
            return

        headers = []
        if provider.value == "Civitai":
            token = get_secret("CIVITAI_TOKEN")
            if token:
                headers = ["--header", "Authorization: Bearer " + token]
            else:
                print("No se encontró el secreto CIVITAI_TOKEN. Se intentará como descarga pública.")
        elif provider.value == "Hugging Face":
            token = get_secret("HF_TOKEN")
            if token:
                headers = ["--header", "Authorization: Bearer " + token]
            else:
                print("No se encontró el secreto HF_TOKEN. Se intentará como descarga pública.")

        print("Descargando en:", dest)
        temp_dest = dest + ".part"
        cmd = ["curl", "-fL", "--retry", "2", "--connect-timeout", "30", "-o", temp_dest]
        if headers:
            cmd += headers
        cmd += [url]
        try:
            result = subprocess.run(cmd, text=True, capture_output=True)
            if result.returncode != 0:
                print("La descarga falló. Comprueba el enlace, los permisos y el token.")
                if result.stderr:
                    print(result.stderr[-1500:])
                if os.path.exists(temp_dest):
                    os.remove(temp_dest)
                return
            if not os.path.exists(temp_dest) or os.path.getsize(temp_dest) == 0:
                print("La descarga produjo un archivo vacío; se descartó.")
                if os.path.exists(temp_dest):
                    os.remove(temp_dest)
                return
            os.replace(temp_dest, dest)
            print("Descarga completada. Tamaño:", round(os.path.getsize(dest) / (1024**2), 2), "MiB")
            print("Destino:", dest)
            print("Nota: comprueba la compatibilidad del recurso con tu modelo base antes de usarlo.")
        except Exception as e:
            print("Error inesperado durante la descarga:", str(e))
            if os.path.exists(temp_dest):
                os.remove(temp_dest)

download_btn.on_click(on_download_clicked)
display(widgets.VBox([provider, resource_type, url_field, filename_field, download_btn, download_out]))


## 4. Iniciar ComfyUI

La siguiente celda instala `cloudflared` y abre un túnel temporal para acceder a la interfaz. El enlace del túnel puede ser accesible por quien lo tenga; no lo compartas públicamente. Detén la celda cuando termines y cierra la sesión de Colab para liberar recursos.


In [ ]:
# 4. Iniciar ComfyUI y mostrar un enlace temporal de acceso
import os
import subprocess
import time
import re
from IPython.display import display, Markdown

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        ["bash", "-lc", "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"],
        check=True
    )

repo = "/content/ComfyUI"
log_path = "/tmp/cloudflared-comfyui.log"
with open(log_path, "w") as log_file:
    comfy_process = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=repo,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT
    )
    print("ComfyUI se está iniciando...")
    time.sleep(10)
    if comfy_process.poll() is not None:
        raise RuntimeError("ComfyUI se cerró al iniciar. Revisa las celdas anteriores.")

    tunnel_process = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

public_url = None
for _ in range(40):
    time.sleep(1)
    try:
        with open(log_path, "r") as f:
            log_text = f.read()
        match = re.search(r"https://[-\\w]+\\.trycloudflare\\.com", log_text)
        if match:
            public_url = match.group(0)
            break
    except OSError:
        pass
    if tunnel_process.poll() is not None:
        break

if public_url:
    display(Markdown("### [Abrir interfaz de ComfyUI](" + public_url + ")"))
    print("No compartas este enlace. Es un túnel temporal.")
else:
    print("No se detectó el enlace automáticamente. Revisa la salida de cloudflared o vuelve a ejecutar esta celda.")


## 5. Solución de problemas

- **No hay GPU:** selecciona una GPU en la configuración del entorno de ejecución, si está disponible en tu cuenta.
- **El modelo no aparece:** verifica que esté en la carpeta correcta dentro de `MyDrive/ComfyUI/models/` y recarga ComfyUI.
- **Error 401/403:** comprueba el token, los permisos y si aceptaste las condiciones del modelo.
- **Poco espacio:** elimina modelos que no uses desde Google Drive y revisa el almacenamiento disponible.
- **La sesión se desconecta:** la GPU de Colab no es permanente; los archivos guardados en Drive se conservan, pero debes volver a ejecutar las celdas de instalación e inicio.

Este notebook no puede garantizar que una GPU gratuita esté disponible ni que todos los enlaces de proveedores funcionen con el mismo método de autenticación.
